# Các dòng thiếu `DomainName` trong ngày 16

Notebook kiểm tra đồng thời dữ liệu Event 4624 và Event 4625. File Parquet được đọc theo từng batch để giới hạn lượng RAM sử dụng. Một giá trị `DomainName` được xem là thiếu nếu là `null`, chuỗi rỗng, hoặc chỉ chứa khoảng trắng.

In [4]:
from pathlib import Path
from typing import Optional

import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

PARQUET_RELATIVE_PATHS = {
    "Event 4624": Path("data/interim/event_4624/event_4624_day-16.parquet"),
    "Event 4625": Path("data/interim/event_4625/event_4625_day-16.parquet"),
}
TARGET_COLUMN = "DomainName"
BATCH_SIZE = 250_000


def find_project_file(relative_path: Path, start_directory: Optional[Path] = None) -> Path:
    """Tìm file từ thư mục hiện tại hoặc một trong các thư mục cha."""
    start_directory = (start_directory or Path.cwd()).resolve()

    for directory in (start_directory, *start_directory.parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy '{relative_path}' từ '{start_directory}' hoặc các thư mục cha."
    )


parquet_paths = {
    event_name: find_project_file(relative_path)
    for event_name, relative_path in PARQUET_RELATIVE_PATHS.items()
}

for event_name, parquet_path in parquet_paths.items():
    print(f"{event_name}: {parquet_path}")

Event 4624: D:\Github Repo\ueba-benchmark\data\interim\event_4624\event_4624_day-16.parquet
Event 4625: D:\Github Repo\ueba-benchmark\data\interim\event_4625\event_4625_day-16.parquet


In [5]:
def missing_text_mask(values: pa.Array) -> pa.Array:
    """Trả về mask cho null, chuỗi rỗng và chuỗi chỉ có khoảng trắng."""
    values_without_null = pc.fill_null(values, "")
    return pc.equal(pc.utf8_trim_whitespace(values_without_null), "")


def read_rows_with_missing_text(
    file_path: Path,
    column: str,
    batch_size: int = 250_000,
) -> pd.DataFrame:
    """Đọc Parquet theo batch và trả về toàn bộ dòng có giá trị text bị thiếu."""
    parquet_file = pq.ParquetFile(file_path)
    column_names = parquet_file.schema_arrow.names

    if column not in column_names:
        raise KeyError(
            f"Không có cột '{column}'. Các cột hiện có: {column_names}"
        )

    column_index = column_names.index(column)
    matching_frames: list[pd.DataFrame] = []

    for batch in parquet_file.iter_batches(batch_size=batch_size):
        mask = missing_text_mask(batch.column(column_index))
        if pc.any(mask).as_py():
            filtered_batch = pa.Table.from_batches([batch]).filter(mask)
            matching_frames.append(filtered_batch.to_pandas())

    if not matching_frames:
        return pd.DataFrame(columns=column_names)

    return pd.concat(matching_frames, ignore_index=True)


In [6]:
missing_rows_by_event = {
    event_name: read_rows_with_missing_text(
        parquet_path,
        TARGET_COLUMN,
        batch_size=BATCH_SIZE,
    )
    for event_name, parquet_path in parquet_paths.items()
}

total_missing = sum(len(rows) for rows in missing_rows_by_event.values())
print(f"Tổng số dòng thiếu {TARGET_COLUMN}: {total_missing:,}")

for event_name, missing_rows in missing_rows_by_event.items():
    print("\n" + "=" * 100)
    print(f"{event_name}: {len(missing_rows):,} dòng thiếu {TARGET_COLUMN}")
    print("=" * 100)

    if missing_rows.empty:
        print("Không có dòng nào bị thiếu.")
    else:
        # to_string() giúp in đủ tất cả dòng và tất cả cột thay vì bị rút gọn.
        print(missing_rows.to_string(index=False))

Tổng số dòng thiếu DomainName: 37

Event 4624: 4 dòng thiếu DomainName
   Time  EventID    LogHost  LogonType LogonTypeDescription   UserName DomainName   LogonID SubjectUserName SubjectDomainName SubjectLogonID Status     Source ServiceName Destination AuthenticationPackage FailureReason  ProcessName ProcessID ParentProcessName ParentProcessID
1312871     4624 Comp455682          5              Service User668687        NaN  0x1c2d96             NaN               NaN            NaN    NaN Comp455682         NaN         NaN             Negotiate           NaN services.exe     0x25c               NaN             NaN
1330181     4624 Comp455682          3              Network User039298        NaN 0x2dd2a46             NaN               NaN            NaN    NaN Comp703658         NaN         NaN                  NTLM           NaN          NaN       NaN               NaN             NaN
1332679     4624 Comp455682          3              Network User831626        NaN 0x31f851c          

In [7]:
def read_unique_values(
    file_path: Path,
    column: str,
    batch_size: int = 250_000,
) -> set[Optional[str]]:
    """Đọc một cột theo batch và trả về mọi giá trị duy nhất, kể cả None."""
    parquet_file = pq.ParquetFile(file_path)

    if column not in parquet_file.schema_arrow.names:
        raise KeyError(f"Không có cột '{column}' trong file '{file_path}'.")

    unique_values: set[Optional[str]] = set()
    for batch in parquet_file.iter_batches(
        columns=[column],
        batch_size=batch_size,
    ):
        unique_values.update(pc.unique(batch.column(0)).to_pylist())

    return unique_values


unique_domains_by_event = {
    event_name: read_unique_values(
        parquet_path,
        TARGET_COLUMN,
        batch_size=BATCH_SIZE,
    )
    for event_name, parquet_path in parquet_paths.items()
}

all_unique_domains = set().union(*unique_domains_by_event.values())
sort_key = lambda value: (value is not None, "" if value is None else value.casefold())

for event_name, domain_values in unique_domains_by_event.items():
    print(f"\n{event_name}: {len(domain_values):,} giá trị DomainName duy nhất")
    for value in sorted(domain_values, key=sort_key):
        print(f"  {value!r}")

print(f"\nCả hai file: {len(all_unique_domains):,} giá trị DomainName duy nhất")
for value in sorted(all_unique_domains, key=sort_key):
    print(f"  {value!r}")


Event 4624: 206 giá trị DomainName duy nhất
  None
  'ActiveDirectory'
  'Comp003403'
  'Comp008736'
  'Comp012316'
  'Comp032135'
  'Comp038987'
  'Comp043604'
  'Comp065969'
  'Comp068186'
  'Comp070627'
  'Comp071864'
  'Comp073704'
  'Comp084657'
  'Comp084803'
  'Comp101143'
  'Comp103224'
  'Comp104967'
  'Comp111577'
  'Comp112411'
  'Comp124638'
  'Comp125736'
  'Comp125927'
  'Comp126780'
  'Comp130715'
  'Comp137959'
  'Comp140632'
  'Comp146253'
  'Comp148386'
  'Comp154413'
  'Comp156630'
  'Comp157301'
  'Comp161235'
  'Comp161745'
  'Comp166150'
  'Comp168348'
  'Comp170209'
  'Comp174549'
  'Comp174601'
  'Comp188794'
  'Comp192174'
  'Comp203856'
  'Comp209234'
  'Comp223071'
  'Comp245383'
  'Comp246582'
  'Comp247211'
  'Comp255390'
  'Comp275281'
  'Comp278269'
  'Comp284248'
  'Comp296705'
  'Comp296766'
  'Comp299935'
  'Comp305017'
  'Comp307946'
  'Comp310553'
  'Comp311509'
  'Comp313247'
  'Comp324399'
  'Comp325647'
  'Comp327612'
  'Comp328815'
  'Comp330510